# Exercise: Can an encoder tell blood cells apart?

Each image is one blood cell. There are **8 types** of cell, like `neutrophil`, `lymphocyte`, and `platelet`. Doctors count these types to check a patient's health.

We used a pretrained encoder, [CLIP](https://huggingface.co/openai/clip-vit-base-patch32) from OpenAI, to turn each image into a row of 512 numbers. It is the same encoder as in lecture 17. Your job is to fit a model on those numbers and see how well it names the cell type.

> Aside: The images come from [BloodMNIST](https://medmnist.com/), a set of small blood cell photos.

## Setup

In [ ]:
library('tidymodels')
library('embed')

options(repr.plot.res = 250, repr.plot.height = 3, repr.plot.width = 5)

## 1. Load the data

Run this cell. It loads the table and gives each number in `label` its cell type name.

In [ ]:
cell_types <- c('basophil', 'eosinophil', 'erythroblast', 'immature granulocyte',
                'lymphocyte', 'monocyte', 'neutrophil', 'platelet')

blood <- readr::read_csv('https://raw.githubusercontent.com/UNC-BIOS-512/jupyterlite/main/data/bloodmnist-emb.csv.gz', show_col_types = FALSE) |>
    mutate(label = factor(label, levels = 0:7, labels = cell_types))

blood |> select(label, dim_1:dim_5) |> head()

In [ ]:
blood |> count(label)

**Check:** how many rows? How many columns? Which cell type is the most common? Which is the rarest?

## 2. Split the data

Run this cell. Do not change it.

- `train`: half of the images. The model learns from these.
- `test`: the other half. The model has never seen these. We measure accuracy here.

In [ ]:
set.seed(512)

split <- initial_split(blood, prop = 0.5, strata = label)
train <- training(split)
test  <- testing(split)

nrow(train)

## 3. Visualize with UMAP

We cannot plot 512 dimensions at once. **UMAP** squeezes all the dimensions into 2 new columns, `UMAP1` and `UMAP2`. Similar images will have similar `UMAP1` and `UMAP2` values.

Run the cell. It makes the 2 UMAP columns for the `train` images. (It takes a few seconds.)

In [ ]:
set.seed(1)

umap_coords <- recipe(label ~ ., data = train) |>
    step_umap(all_predictors(), num_comp = 2) |>
    prep() |>
    bake(new_data = NULL)

umap_coords |> head()

**Your turn:** plot `UMAP1` against `UMAP2`. Color the points by `label`.

In [ ]:
# Chart code goes here

**Check:** do the cell types land in different places?

- Do you see clear groups? Are they perfectly separate?
- Which cell types are hard to tell apart?

## 4. Fit a random forest

Fit a random forest on `train`. Use the formula `label ~ .` (all the `dim_` columns are predictors).

```r
rand_forest(trees = 200) |> set_mode('classification') |> set_engine('ranger')
```

In [ ]:
# Fit your model here

## 5. Evaluate on the test set

Find the accuracy of your model on `test`.

<details>
<summary>Hint: how do I find the accuracy?</summary>

1. `predict()` gives the predicted class for each test image.
2. `bind_cols()` puts it next to the true `label`.
3. `accuracy()` compares them.

```r
predict(fit, new_data = test) |>
    bind_cols(test) |>
    accuracy(truth = label, estimate = .pred_class)
```

Here `fit` is the name of your fitted model. Use your own name.

</details>

In [ ]:
# Accuracy on the test set

**Check:** with 8 types, guessing at random would get about 0.13. A good model gets about 0.80 to 0.85 on the test images. How does yours compare?

- Is the accuracy on `train` higher than on `test`? What does that tell you?

## 6. Bonus: which cells does it mix up?

Accuracy is one number for all 8 types. A **confusion matrix** shows which types the model confuses.

Make one with `conf_mat()`, then look back at your UMAP. Do the types that overlap in the UMAP get mixed up by the model?

In [ ]:
# Confusion matrix here

*Your answer:* Which cell types does the model mix up? Why do you think so?